# Notebook 1 of 2 — Ingest: Internet → ADLS → Bronze

**Your job:** get NYC taxi and weather data from the internet into your lake, keep an untouched copy, then store it as **Bronze Delta** and register it in Unity Catalog.

> **Learning rule:** AI tools are allowed here. Understanding is not optional. You will explain every transformation in your own words in `OBSERVATION_REPORT.md`.

### The naming contract (do not improvise)

| Layer | ADLS path (inside container `lakehouse`) | Unity Catalog table |
|---|---|---|
| Raw taxi files | `raw/taxi/yellow_tripdata_2025-08.parquet` (also `-09`, `-10`) | — |
| Raw weather file | `raw/weather/nyc_daily_2025-08-01_2025-10-31.json` | — |
| Bronze taxi | `bronze/taxi_trips/` | `<catalog>.bronze.taxi_trips` |
| Bronze weather | `bronze/weather_daily/` | `<catalog>.bronze.weather_daily` |

All tables are **external Delta tables** pointing at the same storage account and container.

### Setup
Fill in the two `<...>` values, then run the cell. Use the existing catalog you were given (`SHOW CATALOGS` lists it). You cannot create your own.

In [0]:
# ---- Fill in the two values marked <...> -------------------------------------
STORAGE_ACCOUNT = "dkdedls01"  # your ADLS Gen2 account name
CATALOG = "sona_db_practices"  # run SHOW CATALOGS and use the existing one given to you

# ---- Do NOT change these: the names are part of the contract ------------------
CONTAINER = "lakehouse"
BASE      = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
MONTHS    = ["2025-08", "2025-09", "2025-10"]

TAXI_URL     = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month}.parquet"
WEATHER_FILE = "nyc_daily_2025-08-01_2025-10-31.json"
WEATHER_URL  = ("https://archive-api.open-meteo.com/v1/archive?latitude=40.7128&longitude=-74.006"
                "&start_date=2025-08-01&end_date=2025-10-31"
                "&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,wind_speed_10m_max"
                "&timezone=America%2FNew_York")

---
## Step 1 — Land the raw files in ADLS

**What:** download 3 monthly taxi Parquet files and 1 weather JSON, and put them **unchanged** in the `raw/` folders from the contract table.

**Why:** if your Bronze logic turns out wrong, you rebuild from your own copy instead of hitting the internet again. Raw is the evidence.

| Source | Where it comes from |
|---|---|
| Taxi (3 files, ~60–75 MB each) | `TAXI_URL` with each month from `MONTHS` |
| Weather (daily, NYC, Aug–Oct 2025) | `WEATHER_URL` (Open-Meteo, free, no key) |

**Hints**
- Spark cannot read an `https://` address. Python downloads it; Spark reads it afterwards.
- You are on serverless: download to the driver's local disk (`/tmp`) with `requests`, then move it into the lake with `dbutils.fs.cp("file:/tmp/...", "abfss://...")`.
- Stream the download in chunks (`stream=True`) instead of loading 70 MB into memory. Call `raise_for_status()` so a bad URL fails loudly.
- Delete the temp file afterwards. Use `dbutils.fs.ls` to prove the files arrived and note their sizes.

**Pause and observe →** Why store an untouched copy in `raw/` instead of writing Delta straight from the download? Answer in report §1.

In [0]:
# Step 1: your code here

import os
import requests

response = requests.get(WEATHER_URL, stream=True, timeout=120)
response.raise_for_status()

weather_file = f"/Workspace/Users/sonajyothiputturu@gmail.com/Drafts/{WEATHER_FILE}"

with open(weather_file, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

dbutils.fs.cp(f"file:{weather_file}", f"{BASE}/raw/weather/" + WEATHER_FILE)
os.remove(weather_file)

In [0]:
import os
import requests
from time import sleep

for month in MONTHS:
    print(month)
    TAXI_URL     = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month}.parquet"
    response = requests.get(TAXI_URL, stream=True, timeout=120)
    response.raise_for_status()

    taxi_file = f"/Workspace/Users/sonajyothiputturu@gmail.com/Drafts/yellow_tripdata_{month}.parquet"
    print(taxi_file)
    taxi_file_name = f"yellow_tripdata_{month}.parquet"

    with open(taxi_file, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    sleep(10)
    dbutils.fs.cp(f"file:{taxi_file}", f"{BASE}/raw/taxi/" + taxi_file_name)
    os.remove(taxi_file)
    sleep(5)


2025-08
/Workspace/Users/sonajyothiputturu@gmail.com/Drafts/yellow_tripdata_2025-08.parquet
2025-09
/Workspace/Users/sonajyothiputturu@gmail.com/Drafts/yellow_tripdata_2025-09.parquet
2025-10
/Workspace/Users/sonajyothiputturu@gmail.com/Drafts/yellow_tripdata_2025-10.parquet


---
## Step 2 — Bronze taxi: one Delta table for all three months

**Rules**
1. Read each month's raw Parquet and write to `bronze/taxi_trips/` as **Delta**.
2. Keep the **20 source columns exactly as they arrive** (names, types, values). No renaming, no casting, no filtering. Bronze is the evidence.
3. Add **three metadata columns**: `source_file` (string, the file name), `source_month` (string, `yyyy-MM`), `ingested_at` (timestamp).
4. Partition the table by `source_month`.
5. **Re-runnable:** running the cell twice must give the same row count, not double.

**Hints**
- Loop over `MONTHS`, one month at a time.
- `F.lit(...)` for constants, `F.current_timestamp()` for the load time.
- For rule 5 look at Delta's `replaceWhere` write option (with `mode("overwrite")`): it replaces only the month you are writing.
- Finish with `printSchema()` and a count per `source_month`.

**Pause and observe →** Which column types surprise you (look at `passenger_count`, the two timestamps, `RatecodeID`)? Why would a taxi meter count passengers in a `long`? Note row counts per month in report §2.

In [0]:
# Step 2: your code here
from pyspark.sql import functions as F

bronze_taxi = None
for month in MONTHS:
    file_name = f"yellow_tripdata_{month}.parquet"
    month_df = (
        spark.read.parquet(f"{BASE}/raw/taxi/{file_name}")
        .withColumn("source_file", F.lit(file_name))
        .withColumn("source_month", F.lit(month))
        .withColumn("ingested_at", F.current_timestamp())
    )
    bronze_taxi = month_df if bronze_taxi is None else bronze_taxi.unionByName(month_df)

(bronze_taxi.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("source_month")
    .save(f"{BASE}/bronze/taxi_trips"))

bronze_taxi = spark.read.format("delta").load(f"{BASE}/bronze/taxi_trips")
bronze_taxi.printSchema()
display(bronze_taxi.groupBy("source_month").count().orderBy("source_month"))


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_month: string (nullab

source_month,count
2025-08,3574091
2025-09,4251015
2025-10,4428699


In [0]:
spark.read.format("delta").load(f"{BASE}/bronze/taxi_trips")
df = spark.read.format("delta").load(f"{BASE}/bronze/taxi_trips")
display(df.printSchema())


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- source_month: string (nullable = true)
 |-- current_timestamp: string (

In [0]:
df.groupBy("source_month").count().show()

+------------+-------+
|source_month|  count|
+------------+-------+
|     2025-09|4251015|
|     2025-10|4428699|
|     2025-08|3574091|
+------------+-------+



---
## Step 3 — Bronze weather: flatten the JSON

**Rules**
1. The JSON contains one `daily` object made of **parallel arrays** (`time`, `temperature_2m_max`, …). Turn it into **one row per date**.
2. Keep the API's column names and values as they are. Add `source_file` and `ingested_at`.
3. Write Delta to `bronze/weather_daily/` (overwrite is fine; this one is a single file). Expect **92 rows**.

**Hints**
- JSON that spans many lines needs the `multiLine` read option.
- `select("daily.*")` gives you the arrays as columns. `F.arrays_zip` + `F.explode` turn parallel arrays into rows.

**Pause and observe →** Why is "one object with arrays" an awkward shape for analytics, and what does one row per date make easy? Report §2.

In [0]:
df=spark.read.format("json").load(f"{BASE}/raw/weather")
display(df)

daily,daily_units,elevation,generationtime_ms,latitude,longitude,timezone,timezone_abbreviation,utc_offset_seconds
"List(List(1.0, 0.0, 0.0, 0.0, 0.0, 0.9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.2, 1.0, 0.0, 4.9, 3.6, 0.0, 0.0, 2.6, 0.0, 0.0, 0.0, 0.2, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.1, 3.2, 20.8, 17.4, 0.0, 0.0, 5.0, 0.0, 0.0, 0.7, 0.5, 0.0, 0.0, 6.8, 0.1, 0.0, 0.0, 0.0, 0.0, 6.6, 4.7, 46.9, 0.0, 0.1, 0.6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.4, 0.0, 0.0, 1.4, 15.5, 31.6, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 10.2, 0.1, 1.5, 0.1, 0.0, 0.0, 0.1, 0.0, 0.0, 0.1, 44.4, 0.0), List(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0), List(23.0, 25.7, 28.1, 33.2, 31.0, 28.5, 28.6, 28.2, 30.0, 33.3, 32.5, 33.4, 32.0, 31.3, 32.5, 30.6, 35.3, 24.0, 24.2, 22.7, 21.9, 26.9, 28.0, 26.3, 31.7, 27.6, 25.0, 27.1, 27.5, 25.8, 27.5, 26.7, 26.9, 26.4, 25.2, 28.5, 30.4, 20.8, 21.7, 22.3, 20.7, 26.9, 27.5, 26.5, 29.8, 28.6, 24.5, 21.1, 27.4, 28.1, 23.8, 22.1, 22.7, 29.5, 26.7, 24.9, 26.6, 24.4, 28.0, 24.1, 25.4, 20.3, 19.1, 22.2, 27.9, 29.9, 26.9, 27.4, 21.5, 15.1, 16.2, 19.0, 16.5, 14.9, 16.2, 19.9, 15.3, 17.1, 17.9, 20.8, 20.1, 19.3, 16.7, 15.1, 14.9, 14.5, 14.4, 13.0, 12.4, 13.6, 16.5, 13.6), List(17.8, 17.1, 16.5, 17.0, 19.9, 20.3, 19.0, 17.0, 17.3, 18.3, 20.8, 21.1, 23.5, 22.6, 22.8, 22.4, 21.8, 18.6, 17.5, 15.9, 15.7, 14.9, 17.2, 17.5, 19.1, 17.6, 16.5, 15.1, 17.8, 14.2, 14.9, 16.6, 16.5, 15.4, 16.0, 18.1, 18.5, 16.7, 13.0, 13.1, 15.2, 15.3, 16.2, 16.1, 17.0, 17.8, 17.4, 17.2, 18.2, 18.0, 14.5, 13.6, 11.9, 16.7, 18.3, 19.8, 18.6, 16.9, 17.7, 17.2, 17.0, 12.1, 9.6, 8.9, 11.7, 14.4, 14.6, 17.1, 12.9, 7.6, 4.8, 9.6, 13.0, 12.2, 12.2, 10.4, 7.2, 7.2, 6.4, 9.9, 10.9, 8.0, 10.6, 8.5, 5.8, 5.8, 6.8, 3.4, 4.2, 7.1, 10.8, 9.4), List(2025-08-01, 2025-08-02, 2025-08-03, 2025-08-04, 2025-08-05, 2025-08-06, 2025-08-07, 2025-08-08, 2025-08-09, 2025-08-10, 2025-08-11, 2025-08-12, 2025-08-13, 2025-08-14, 2025-08-15, 2025-08-16, 2025-08-17, 2025-08-18, 2025-08-19, 2025-08-20, 2025-08-21, 2025-08-22, 2025-08-23, 2025-08-24, 2025-08-25, 2025-08-26, 2025-08-27, 2025-08-28, 2025-08-29, 2025-08-30, 2025-08-31, 2025-09-01, 2025-09-02, 2025-09-03, 2025-09-04, 2025-09-05, 2025-09-06, 2025-09-07, 2025-09-08, 2025-09-09, 2025-09-10, 2025-09-11, 2025-09-12, 2025-09-13, 2025-09-14, 2025-09-15, 2025-09-16, 2025-09-17, 2025-09-18, 2025-09-19, 2025-09-20, 2025-09-21, 2025-09-22, 2025-09-23, 2025-09-24, 2025-09-25, 2025-09-26, 2025-09-27, 2025-09-28, 2025-09-29, 2025-09-30, 2025-10-01, 2025-10-02, 2025-10-03, 2025-10-04, 2025-10-05, 2025-10-06, 2025-10-07, 2025-10-08, 2025-10-09, 2025-10-10, 2025-10-11, 2025-10-12, 2025-10-13, 2025-10-14, 2025-10-15, 2025-10-16, 2025-10-17, 2025-10-18, 2025-10-19, 2025-10-20, 2025-10-21, 2025-10-22, 2025-10-23, 2025-10-24, 2025-10-25, 2025-10-26, 2025-10-27, 2025-10-28, 2025-10-29, 2025-10-30, 2025-10-31), List(16.1, 10.1, 11.1, 14.6, 9.7, 13.0, 13.8, 12.8, 11.3, 13.0, 12.7, 14.1, 13.3, 6.5, 13.1, 10.5, 14.6, 16.8, 13.1, 15.1, 22.6, 13.3, 21.6, 18.6, 19.4, 16.6, 15.6, 14.1, 16.0, 12.7, 15.1, 16.8, 13.9, 13.8, 20.4, 15.7, 19.0, 9.4, 9.2, 12.8, 13.8, 13.6, 11.4, 12.7, 9.3, 12.1, 17.0, 14.4, 7.7, 11.1, 12.3, 11.2, 11.2, 14.9, 16.9, 15.7, 8.7, 6.1, 6.5, 10.7, 14.1, 14.4, 8.7, 8.9, 9.4, 9.6, 12.8, 19.2, 17.4, 14.9, 11.4, 14.0, 24.6, 28.6, 18.5, 13.4, 17.5, 15.0, 7.6, 20.5, 23.9, 12.7, 21.1, 18.1, 11.9, 9.2, 7.2, 11.7, 21.7, 16.1, 26.5, 27.4))","List(mm, cm, °C, °C, iso8601, km/h)",32.0,0.5921125411987305,40.738136,-74.04254,America/New_York,GMT-5,-18000


In [0]:
df.select("daily.*").show()

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|   precipitation_sum|        snowfall_sum|  temperature_2m_max|  temperature_2m_min|                time|  wind_speed_10m_max|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|[1.0, 0.0, 0.0, 0...|[0.0, 0.0, 0.0, 0...|[23.0, 25.7, 28.1...|[17.8, 17.1, 16.5...|[2025-08-01, 2025...|[16.1, 10.1, 11.1...|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+



In [0]:
from pyspark.sql import functions as F

df = spark.read.option("multiLine", True).json(f"{BASE}/raw/weather/{WEATHER_FILE}")
df = df.select("daily.*")
df = df.select(
    F.explode(
        F.arrays_zip(
            "time",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "snowfall_sum",
            "wind_speed_10m_max"
        )
    ).alias("z")
).select("z.*")

df = (
    df.withColumn("source_file", F.lit(WEATHER_FILE))
      .withColumn("ingested_at", F.current_timestamp())
)

(df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{BASE}/bronze/weather_daily"))

df = spark.read.format("delta").load(f"{BASE}/bronze/weather_daily")
display(df.orderBy("time"))



time,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,wind_speed_10m_max,source_file,ingested_at
2025-08-01,23.0,17.8,1.0,0.0,16.1,nyc_daily_2025-08-01_2025-10-31.json,2026-11-12T16:15:58.295Z
2025-08-02,25.7,17.1,0.0,0.0,10.1,nyc_daily_2025-08-01_2025-10-31.json,2026-11-12T16:15:58.295Z
2025-08-03,28.1,16.5,0.0,0.0,11.1,nyc_daily_2025-08-01_2025-10-31.json,2026-11-12T16:15:58.295Z
2025-08-04,33.2,17.0,0.0,0.0,14.6,nyc_daily_2025-08-01_2025-10-31.json,2026-11-12T16:15:58.295Z
2025-08-05,31.0,19.9,0.0,0.0,9.7,nyc_daily_2025-08-01_2025-10-31.json,2026-11-12T16:15:58.295Z
2025-08-06,28.5,20.3,0.9,0.0,13.0,nyc_daily_2025-08-01_2025-10-31.json,2026-11-12T16:15:58.295Z
2025-08-07,28.6,19.0,0.0,0.0,13.8,nyc_daily_2025-08-01_2025-10-31.json,2026-11-12T16:15:58.295Z
2025-08-08,28.2,17.0,0.0,0.0,12.8,nyc_daily_2025-08-01_2025-10-31.json,2026-11-12T16:15:58.295Z
2025-08-09,30.0,17.3,0.0,0.0,11.3,nyc_daily_2025-08-01_2025-10-31.json,2026-11-12T16:15:58.295Z
2025-08-10,33.3,18.3,0.0,0.0,13.0,nyc_daily_2025-08-01_2025-10-31.json,2026-11-12T16:15:58.295Z


---
## Step 4 — Register the Bronze external tables

Create schema `bronze` in your catalog, then register `taxi_trips` and `weather_daily` as **external Delta tables** over the folders from Steps 2 and 3 (you already know how). Print `Type`, `Provider` and `Location` for each.

**Hint:** if `CREATE SCHEMA` complains about managed storage, tell your instructor. Do not work around it by writing data somewhere else.

**Pause and observe →** Paste the `DESCRIBE EXTENDED` evidence in report §4. If you ran `DROP TABLE` on an external table, what happens to the files in ADLS? Answer from understanding; **do not test it on your data.**

In [0]:

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG} MANAGED LOCATION '{BASE}/managed/{CATALOG}'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.taxi_trips
USING DELTA
LOCATION '{BASE}/bronze/taxi_trips'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.weather_daily
USING DELTA
LOCATION '{BASE}/bronze/weather_daily'
""")

for table_name in [f"{CATALOG}.bronze.taxi_trips", f"{CATALOG}.bronze.weather_daily"]:
    desc = spark.sql(f"DESCRIBE EXTENDED {table_name}")
    desc.filter(desc.col_name.isin("Type", "Provider", "Location")).show(truncate=False)



+--------+------------------------------------------------------------------+-------+
|col_name|data_type                                                         |comment|
+--------+------------------------------------------------------------------+-------+
|Type    |EXTERNAL                                                          |       |
|Location|abfss://lakehouse@dkdedls01.dfs.core.windows.net/bronze/taxi_trips|       |
|Provider|delta                                                             |       |
+--------+------------------------------------------------------------------+-------+

+--------+---------------------------------------------------------------------+-------+
|col_name|data_type                                                            |comment|
+--------+---------------------------------------------------------------------+-------+
|Type    |EXTERNAL                                                             |       |
|Location|abfss://lakehouse@dkdedls01.dfs

---
## Step 5 — Validate, then prove it is re-runnable

Run the check cell below (it is given). Fix anything marked ❌. Then **re-run your Step 2 cell** and run the checks again. Record the counts before and after in report §3.

> **Git checkpoint:** export or commit this notebook now (`git commit -m "notebook 1: bronze ingestion"`).

In [0]:
from pyspark.sql import functions as F

_results = []
def check(label, ok):
    _results.append(bool(ok))
    print(("✅" if ok else "❌"), label)

def table_info(name):
    return {r.col_name: r.data_type for r in spark.sql(f"DESCRIBE EXTENDED {name}").collect()}

def summary():
    print(f"\n{sum(_results)} of {len(_results)} checks passed")

# ---------------- RAW ----------------
raw_taxi = sorted(f.name for f in dbutils.fs.ls(f"{BASE}/raw/taxi"))
check("raw/taxi holds exactly the 3 monthly parquet files",
      raw_taxi == [f"yellow_tripdata_{m}.parquet" for m in MONTHS])
check("raw/weather holds the weather JSON",
      [f.name for f in dbutils.fs.ls(f"{BASE}/raw/weather")] == [WEATHER_FILE])

# ---------------- BRONZE TAXI ----------------
t = f"{CATALOG}.bronze.taxi_trips"
info = table_info(t)
check("bronze.taxi_trips is an EXTERNAL table", info.get("Type") == "EXTERNAL")
check("bronze.taxi_trips points at <container>/bronze/taxi_trips", info.get("Location", "").rstrip("/").endswith("/bronze/taxi_trips"))
check("bronze.taxi_trips is Delta", str(info.get("Provider", "")).lower() == "delta")

bt = spark.table(t)
src_cols = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance",
            "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID", "payment_type", "fare_amount",
            "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
            "congestion_surcharge", "Airport_fee", "cbd_congestion_fee"]
check("Bronze keeps the 20 source column names untouched, plus 3 metadata columns",
      set(bt.columns) == set(src_cols + ["source_file", "source_month", "ingested_at"]))

per_month = {r.source_month: r["count"] for r in bt.groupBy("source_month").count().collect()}
raw_counts = {m: spark.read.parquet(f"{BASE}/raw/taxi/yellow_tripdata_{m}.parquet").count() for m in MONTHS}
check("Bronze rows per month == rows in the raw file (no loss, no duplicates)", per_month == raw_counts)

# ---------------- BRONZE WEATHER ----------------
w = f"{CATALOG}.bronze.weather_daily"
winfo = table_info(w)
check("bronze.weather_daily is an EXTERNAL table at <container>/bronze/weather_daily",
      winfo.get("Type") == "EXTERNAL" and winfo.get("Location", "").rstrip("/").endswith("/bronze/weather_daily"))
bw = spark.table(w)
check("weather: 92 rows (Aug 1 - Oct 31) and 92 distinct dates", bw.count() == 92 and bw.select("time").distinct().count() == 92)
check("weather keeps the API column names",
      {"time", "temperature_2m_max", "temperature_2m_min", "precipitation_sum", "snowfall_sum",
       "wind_speed_10m_max", "source_file", "ingested_at"} == set(bw.columns))

summary()
print("Bronze rows per month:", dict(sorted(per_month.items())))

✅ raw/taxi holds exactly the 3 monthly parquet files
✅ raw/weather holds the weather JSON
✅ bronze.taxi_trips is an EXTERNAL table
✅ bronze.taxi_trips points at <container>/bronze/taxi_trips
✅ bronze.taxi_trips is Delta
✅ Bronze keeps the 20 source column names untouched, plus 3 metadata columns
✅ Bronze rows per month == rows in the raw file (no loss, no duplicates)
✅ bronze.weather_daily is an EXTERNAL table at <container>/bronze/weather_daily
✅ weather: 92 rows (Aug 1 - Oct 31) and 92 distinct dates
✅ weather keeps the API column names

10 of 10 checks passed
Bronze rows per month: {'2025-08': 3574091, '2025-09': 4251015, '2025-10': 4428699}
